In [5]:
%pip install -q tensorflow scikit-learn seaborn matplotlib pandas numpy sastrawi nltk wordcloud

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers

import datetime as dt
import re  # Modul untuk bekerja dengan ekspresi reguler
import string  # Berisi konstanta string, seperti tanda baca
from nltk.tokenize import word_tokenize  # Tokenisasi teks
from nltk.corpus import stopwords  # Daftar kata-kata berhenti dalam teks

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory  # Menghapus kata-kata berhenti dalam bahasa Indonesia

from wordcloud import WordCloud  # Membuat visualisasi berbentuk awan kata (word cloud) dari teks

import nltk  # Import pustaka NLTK (Natural Language Toolkit).
nltk.download('punkt')  # Mengunduh dataset yang diperlukan untuk tokenisasi teks.
nltk.download('punkt_tab')  # Kompatibel untuk NLTK versi baru yang membutuhkan resource punkt_tab.
nltk.download('stopwords')  # Mengunduh dataset yang berisi daftar kata-kata berhenti (stopwords) dalam berbagai bahasa.

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
# print("TensorFlow version:", tf.__version__)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
DATA_PATH = Path("content/reviews_duolingo.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "File /content/reviews_duolingo.csv belum ditemukan. Jalankan notebook scrapping.ipynb terlebih dahulu."
    )

df = pd.read_csv(DATA_PATH)
print("Shape data awal:", df.shape)
df.head()

Shape data awal: (19819, 9)


,review_id,app_id,app_name,user_name,score,content,thumbs_up,at,app_version
0,7852f726-11f1-4de3-8bab-1add6f09d7f7,com.duolingo,DUOLINGO,Farhan Abdillah,5,gak ada update aku versi old,0,2026-04-26 13:24:18,5.121.14
1,4ba770cc-55dc-41cc-bd78-04524bf94c98,com.duolingo,DUOLINGO,Devi Melani,3,"Why is this app so slow? Previously, I was studying smoothly without any problems, but now, just opening a single le...",0,2026-04-26 13:11:49,NaN
2,3652eaa4-5f07-4c5f-b79a-b6bcb9a1bbb6,com.duolingo,DUOLINGO,Nun_azzahh,5,"Duolingo ngebantu banget , asalnya aku main duolingo main belajar di duolingo cuman buat b inggris trs aku ikut esku...",0,2026-04-26 12:49:42,6.71.2
3,435b2c23-4849-41e6-9f65-54117fc4950c,com.duolingo,DUOLINGO,Sukarti 12,5,bagus sih ini saya jadi bisa belajar bahasa Inggris tapi ada yang aneh pas aku pake tulisan premium malah salah koca...,0,2026-04-26 11:44:12,6.76.3
4,be0eb54e-bd00-4e93-b420-4ed34ca9682c,com.duolingo,DUOLINGO,Alleya zevania Kiara,5,bagus buat lagi yang belajar bahasa asing,0,2026-04-26 11:40:01,5.141.11


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19819 entries, 0 to 19818
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    19819 non-null  object
 1   app_id       19819 non-null  object
 2   app_name     19819 non-null  object
 3   user_name    19819 non-null  object
 4   score        19819 non-null  int64 
 5   content      19819 non-null  object
 6   thumbs_up    19819 non-null  int64 
 7   at           19819 non-null  object
 8   app_version  18165 non-null  object
dtypes: int64(2), object(7)
memory usage: 1.4+ MB


In [14]:
clean_df = df.dropna()
clean_df = clean_df.drop_duplicates()

clean_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18165 entries, 0 to 19818
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    18165 non-null  object
 1   app_id       18165 non-null  object
 2   app_name     18165 non-null  object
 3   user_name    18165 non-null  object
 4   score        18165 non-null  int64 
 5   content      18165 non-null  object
 6   thumbs_up    18165 non-null  int64 
 7   at           18165 non-null  object
 8   app_version  18165 non-null  object
dtypes: int64(2), object(7)
memory usage: 1.4+ MB


In [15]:
display(clean_df["score"].value_counts().sort_index())

score
1      408
2      153
3      421
4     1833
5    15350
Name: count, dtype: int64

In [16]:
min_count = df['score'].value_counts().min()

balanced_df = (
    df.groupby('score', group_keys=False)
      .apply(lambda x: x.sample(min_count, random_state=42))
      .reset_index(drop=True)
)

display(balanced_df["score"].value_counts().sort_index())

C:\Users\DELL\AppData\Local\Temp\ipykernel_17224\3199804696.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min_count, random_state=42))


score
1    206
2    206
3    206
4    206
5    206
Name: count, dtype: int64

In [ ]:
def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

def casefoldingText(text): # Mengubah semua karakter dalam teks menjadi huruf kecil
    text = text.lower()
    return text

def tokenizingText(text): # Memecah atau membagi string, teks menjadi daftar token
    text = word_tokenize(text, preserve_line=True)
    return text

def filteringText(text): # Menghapus stopwords dalam teks
    listStopwords = set(stopwords.words('indonesian'))
    listStopwords1 = set(stopwords.words('english'))
    listStopwords.update(listStopwords1)
    listStopwords.update(['iya','yaa','gak','nya','na','sih','ku',"di","ga","ya","gaa","loh","kah","woi","woii","woy"])
    filtered = []
    for txt in text:
        if txt not in listStopwords:
            filtered.append(txt)
    text = filtered
    return text

def stemmingText(text): # Mengurangi kata ke bentuk dasarnya yang menghilangkan imbuhan awalan dan akhiran atau ke akar kata
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    # Memecah teks menjadi daftar kata
    words = text.split()

    # Menerapkan stemming pada setiap kata dalam daftar
    stemmed_words = [stemmer.stem(word) for word in words]

    # Menggabungkan kata-kata yang telah distem
    stemmed_text = ' '.join(stemmed_words)

    return stemmed_text

def toSentence(list_words): # Mengubah daftar kata menjadi kalimat
    sentence = ' '.join(word for word in list_words)
    return sentence

In [18]:
slangwords = {"@": "di", "abis": "habis", "wtb": "beli", "masi": "masih", "wts": "jual", "wtt": "tukar", "bgt": "banget", "maks": "maksimal"}
def fix_slangwords(text):
    words = text.split()
    fixed_words = []

    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)

    fixed_text = ' '.join(fixed_words)
    return fixed_text

In [19]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
balanced_df['text_clean'] = balanced_df['content'].apply(cleaningText)

# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
balanced_df['text_casefoldingText'] = balanced_df['text_clean'].apply(casefoldingText)

# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
balanced_df['text_slangwords'] = balanced_df['text_casefoldingText'].apply(fix_slangwords)

# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
balanced_df['text_tokenizingText'] = balanced_df['text_slangwords'].apply(tokenizingText)

# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
balanced_df['text_stopword'] = balanced_df['text_tokenizingText'].apply(filteringText)

# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
balanced_df['text_akhir'] = balanced_df['text_stopword'].apply(toSentence)

LookupError: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - 'C:\\Users\\DELL/nltk_data'
    - 'c:\\Users\\DELL\\AppData\\Local\\Programs\\Python\\Python312\\nltk_data'
    - 'c:\\Users\\DELL\\AppData\\Local\\Programs\\Python\\Python312\\share\\nltk_data'
    - 'c:\\Users\\DELL\\AppData\\Local\\Programs\\Python\\Python312\\lib\\nltk_data'
    - 'C:\\Users\\DELL\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************
